In [1]:
import re
import json
import numpy as np
from enum import Enum
from dataclasses import dataclass
from typing import List, Dict
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class BlockType(Enum):
    PARAGRAPH = "paragraph"
    TABLE = "table"
    LIST = "list"
    HEADING = "heading"
    MIXED = "mixed"

@dataclass
class Block:
    type: BlockType
    text: str

print("Loading embedding model...")
embedder = SentenceTransformer('BAAI/bge-base-en-v1.5')

def clean_artifacts(text: str) -> str:
    return text

def parse_blocks(text: str) -> List[Block]:
    table_pattern = re.compile(r'(<table[\s\S]*?</table>)', re.IGNORECASE)
    parts = table_pattern.split(text)
    
    blocks = []
    list_pattern = re.compile(r'^(\s*[-*+]\s+|\s*\d+\.\s+|\s*\([a-zA-Z0-9]{1,4}\)\s+)')
    
    for part in parts:
        part = part.strip()
        if not part: continue
            
        if part.lower().startswith('<table'):
            blocks.append(Block(BlockType.TABLE, part))
        else:
            sub_parts = re.split(r'\n{2,}', part)
            for sp in sub_parts:
                sp = sp.strip()
                if not sp: continue
                
                if re.match(r'^#{1,6}\s+', sp):
                    blocks.append(Block(BlockType.HEADING, sp))
                elif list_pattern.match(sp):
                    blocks.append(Block(BlockType.LIST, sp))
                else:
                    blocks.append(Block(BlockType.PARAGRAPH, sp))
                    
    return blocks

def split_list_by_length(list_text: str, max_chars: int) -> List[str]:
    if len(list_text) <= max_chars:
        return [list_text]
        
    marker_pattern = re.compile(r'(^|\n)(\s*[-*+]\s+|\s*\d+\.\s+|\s*\([a-zA-Z0-9]{1,4}\)\s+)')
    parts = marker_pattern.split(list_text)
    
    items, current_item = [], ""
    for part in parts:
        if marker_pattern.match(part):
            if current_item: items.append(current_item.strip())
            current_item = part
        else:
            current_item += part
    if current_item: items.append(current_item.strip())
        
    chunks, current_chunk, current_len = [], [], 0
    for item in items:
        if current_len + len(item) > max_chars and current_chunk:
            chunks.append("\n".join(current_chunk))
            current_chunk = [item]
            current_len = len(item)
        else:
            current_chunk.append(item)
            current_len += len(item)
            
    if current_chunk:
        chunks.append("\n".join(current_chunk))
    return chunks

def calculate_similarity(text1: str, text2: str) -> float:
    if not text1 or not text2: return 0.0
    # Compare document blocks in BGE's document embedding space.
    emb1 = embedder.encode(text1).reshape(1, -1)
    emb2 = embedder.encode(text2).reshape(1, -1)
    return cosine_similarity(emb1, emb2)[0][0]

def semantic_chunker(text: str, sim_threshold: float = 0.4, 
                     max_chars: int = 1600, min_chars: int = 800) -> List[Dict[str, str]]:
    
    text = clean_artifacts(text)
    
    if len(text) <= max_chars:
        return [{"type": "mixed", "text": text}]
        
    raw_blocks = parse_blocks(text)
    processed_blocks = []
    pending_heading = None
    
    # Pre-processing: Attach headings, group contiguous lists
    for block in raw_blocks:
        if block.type == BlockType.HEADING:
            pending_heading = (pending_heading + "\n\n" + block.text) if pending_heading else block.text
            continue
            
        if pending_heading:
            block.text = pending_heading + "\n\n" + block.text
            pending_heading = None
            
        if processed_blocks and block.type == BlockType.LIST and processed_blocks[-1].type == BlockType.LIST:
            processed_blocks[-1].text += "\n\n" + block.text
        else:
            processed_blocks.append(block)

    final_chunks = []
    current_chunk = []
    current_length = 0
    current_type = None

    def flush_chunk():
        nonlocal current_chunk, current_length, current_type
        if current_chunk:
            final_chunks.append({
                "type": current_type.value if current_type else "mixed",
                "text": "\n\n".join(current_chunk)
            })
            current_chunk = []
            current_length = 0
            current_type = None

    for block in processed_blocks:
        block_len = len(block.text)
        
        if block_len > max_chars:
            flush_chunk()
            if block.type == BlockType.LIST:
                for l in split_list_by_length(block.text, max_chars):
                    final_chunks.append({"type": "list", "text": l})
            else:
                final_chunks.append({"type": block.type.value, "text": block.text})
            continue

        if not current_chunk:
            current_chunk.append(block.text)
            current_length = block_len
            current_type = block.type
        else:
            last_text = current_chunk[-1]
            similarity = 0.0
            if current_type == BlockType.PARAGRAPH and block.type == BlockType.PARAGRAPH:
                similarity = calculate_similarity(last_text, block.text)

            is_under_min = (current_length + block_len) <= min_chars
            
            if (current_type == BlockType.TABLE and block.type != BlockType.TABLE) or \
               (block.type == BlockType.TABLE and current_type != BlockType.TABLE):
                is_under_min = False

            is_semantic_match = (current_type == block.type) and (similarity >= sim_threshold)
            is_tiny_block = block_len < 150 or len(last_text) < 150
            is_structural_packing = (current_type == block.type) and (current_type in (BlockType.TABLE, BlockType.LIST))

            if (is_under_min or is_semantic_match or is_tiny_block or is_structural_packing) and (current_length + block_len) <= max_chars:
                current_chunk.append(block.text)
                current_length += block_len + 2
                if current_type != block.type:
                    current_type = BlockType.MIXED 
            else:
                flush_chunk()
                current_chunk.append(block.text)
                current_length = block_len
                current_type = block.type

    flush_chunk() 
    return final_chunks

def encode_text(text: str) -> List[float]:
    emb = embedder.encode(text)
    return emb.tolist()

# --- Execution pipeline ---
with open("../data/chunks/parentchunks2.json", "r", encoding="utf-8") as f:
    data = json.load(f)

def count_tokens(text: str) -> int:
    if not text:
        return 0
    return len(re.findall(r"\w+|[^\w\s]", text))

children_chunks = []

for parent in data:
    c = count_tokens(parent["text"]) 
    print(c)
    if c <= 400:
        children_chunks.append({
            "parent_id": parent["parent_id"],
            "child_id": f"{parent['parent_id']}_child_1",
            "doc_id": parent.get("doc_id", ""),
            "heading_path": parent.get("heading_path", []), 
            "text": parent["text"],
            "type": "mixed",
            "embedding": encode_text(parent["text"])
        })
    else:
        chunks = semantic_chunker(parent["text"])
        
        for index, chunk in enumerate(chunks, 1):
            children_chunks.append({
                "parent_id": parent["parent_id"],
                "child_id": f"{parent['parent_id']}_child_{index}",
                "doc_id": parent.get("doc_id", ""),
                "heading_path": parent.get("heading_path", []), 
                "text": chunk["text"],
                "type": chunk.get("type", "mixed"),
                "embedding": encode_text(chunk["text"])
            })

# Save output
with open("../data/chunks/children_chunks.json", "w", encoding="utf-8") as f:
    json.dump(children_chunks, f, indent=4, ensure_ascii=False)

C:\Users\Raj Raman\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model...


<All keys matched successfully>


13
34
173
95
104
95
24
54
54
29
62
48
58
93
65
239
306
46
101
143
56
29
51
42
51
27
292
334
4
213
81
192
38
229
277
157
97
140
85
77
126
47
290
311
87
176
137
142
102
120
20
295
117
35
54
52
54
132
90
51
23
524
70
37
20
123
166
50
131
298
361
395
408
306
131
439
475
316
536
445
208
325
138
260
254
133
370
429
471
408
371
391
435
582
295
14
821
2444
792
83
178
169
144
1425
413
529
2735
393
306
1364
495
472
492
138
646
271
2385
2482
575
367
311
71
979
768
158
175
2243
570
1423
103
366
273
696
429
1702
1251
263
1042
316
1600
405
815
1652
126
213
712
926
415
784
467
3391
3055
335
263
413
410
399
1292
632
991
1081
1071
110
162
428
135
216
647
616
898
476
890
837
61
436
415
492
554
58
347
100
506
902
685
502
466
435
294
153
144
349
470
1993
1117
375
779
967
1300
1941
924
477
861
284
669
171
135
588
156
216
334
505
823
420
351
493
1481
404
1820
2341
302
215
453
989
1328
624
456
211
637
374
890
396
444
354
483
534
484
1235
770
2473
370
277
282
319
330
9
746
507
129
223
152
188
103
158
9
129
10